# SysMLDocGen MDK — 验收演示

演示 Jupyter MDK 三大核心功能：**API 客户端**、**双向同步引擎**、**DocGen 模板语言**。

---
## 第一步：初始化

In [ ]:
%load_ext sysmldocgen_mdk

---
## 第二步：连接与浏览

> 一行命令连接系统，查看项目列表和模型列表
> 对比：浏览器需要点菜单→进项目→翻页

In [ ]:
%connect http://localhost:8000 admin admin123

In [ ]:
%list_projects

In [ ]:
%list_models --project 4

---
## 第三步：加载模型数据

> 一键拉取模型全部元素到 Jupyter，自动转 DataFrame
> 展示：表格显示、类型统计、灵活筛选

In [ ]:
%load_model 2

In [ ]:
from sysmldocgen_mdk.jupyter import display_table, display_stats

display_table(model_2_elements)
display_stats(model_2_elements)

In [ ]:
# DataFrame 灵活筛选：只看需求
reqs = model_2_elements[model_2_elements['element_type'] == 'Requirement']
reqs[['element_id', 'element_name', 'description']]

---
## 第四步：双向同步 — 修改与推送

> 离线修改→变更跟踪→差异对比→批量推送
> 核心价值：浏览器改一条存一条，MDK 可批量改、检冲突、一次性推

In [ ]:
# 4.1 拉取快照
%pull 2

In [ ]:
# 4.2 修改元素（离线操作，不依赖网络）
snapshot_2.edit_element('req_1',
    description='[已审核] 用户可通过手机远程控制家中设备')
snapshot_2.edit_element('req_2',
    description='[已审核] 系统应支持定时任务和场景联动')

print('修改了', len(snapshot_2.local_changes), '个元素')
for c in snapshot_2.local_changes:
    print('  -', c.element_id, '的', c.field, ':', str(c.old_value)[:20], '->', str(c.new_value)[:20])

In [ ]:
# 4.3 差异对比（检测冲突）
diff = session_2.diff(snapshot_2)
print('冲突:', diff.has_conflicts)
print('服务端新增:', len(diff.server_only))
print('本地变更:', len(diff.local_only))
print('冲突数:', len(diff.conflicts))

In [ ]:
# 4.4 推送同步
%sync_push 2

In [ ]:
# 4.5 验证：重新拉取确认修改已生效
from sysmldocgen_mdk import Client
updated = Client('http://localhost:8000', 'admin', 'admin123')
updated.get_element(2, 'req_1').get('description')

---
## 第五步：DocGen 模板语言

> 编写模板→即时预览→SysML 专用过滤器
> 对比：浏览器只能在编辑器写模板，提交后才能看效果

In [ ]:
%%render_doc 需求规格说明书
<h1 style="text-align:center">{{ model_name }} 需求规格说明书</h1>
<p style="text-align:center;color:#666">{{ generate_time }} | 共 {{ element_count }} 个元素</p>

---

## 需求清单
{{ requirement_list | req_table("系统需求") }}

## 系统架构
{{ elements | block_hierarchy }}

## 接口定义
{{ elements | iface_matrix }}

In [ ]:
%%render_doc 元素属性
{% for elem in elements[:3] %}
### {{ elem.element_name }}
- 类型: {{ elem.element_type }}
- 属性: {{ elem.attributes | format_attrs }}

{% endfor %}

---
## 第六步：推送模板到服务端并生成文档

> 模板写好后保存到服务端，供团队使用或批量文档生成

In [ ]:
from sysmldocgen_mdk import DocGenTemplate

tpl = DocGenTemplate(
    name="验收演示模板",
    content="""<h1>{{ model_name }} 文档</h1>
<p>版本 {{ model_version }} | {{ generate_time }}</p>

## 需求
{{ requirement_list | req_table("需求") }}

## 架构
{{ elements | block_hierarchy }}
"""
)
result = tpl.to_server(Client('http://localhost:8000', 'admin', 'admin123'), template_type='docgen')
print('模板已保存, ID:', result['id'])

In [ ]:
# 用模板生成文档
doc = Client('http://localhost:8000', 'admin', 'admin123').generate_document(
    project_id=4,
    model_id=2,
    template_id=result['id']
)
print('文档已生成, ID:', doc['id'])

---
## 对比总结

| 操作 | 浏览器 | MDK (Jupyter) |
|------|--------|---------------|
| 查看模型 | 点菜单→进项目→翻页 | `%load_model 2` + DataFrame |
| 统计 | 人工计数 | `display_stats()` 自动画图 |
| 修改元素 | 逐条点开→编辑→保存 | 批量修改 + 变更跟踪 |
| 冲突处理 | 无 | `diff()` 自动检测 |
| 写文档模板 | 编辑器→提交才能看 | `%%render_doc` 即时预览 |